# Scheduled Jobs   

There are 3 Jobs that runs on an automated schedule to keep the Database refreshed.

1. Refresh Trakt Table  (Daily)
1. Full Refresh of TMDB Tables (Weekly)
1. Delta Refresh of TMDB Tables (Daily)
  
    
These Jobs are managed with 2 "manager" python modules under the project scheduler folder   
[tmdb_manager.py](../scheduler/tmdb_manager.py)  (Full and Delta refresh of TMDB data)   
[trakt_manager.py](../scheduler/trakt_manager.py) (Refresh of TRAKT Data)   
  
These are both OOP-based classes with "public" and "private" methods*
  * Python does fully support OOP and conventions for public/private. But the use of an underscore as a convention for flagging "Private" Methods
  
  ![alt text](../images/scheduled_jobs.png)   
  
   

***
### Refresh Trakt Data  
The python module [trakt_manager.py](../scheduler/trakt_manager.py) is responsible for getting the latest trakt data from the API and save it to the database.  

It is responsible for ...
* connecting to the MySQL database using configuration from config.py  
* retrieving Trakt authentication details from the database.  
* calling the Trakt API to get watched episodes and ratings.  
* converting the returned JSON into Pandas DataFrames and merges watched and rating data together.  
* saving the final dataset to the database using dao_trakt.  

It also includes logging and a test method for checking database/auth connectivity.  

It is scheduled to run automatically each day at midnight.  
With this, as I watch shows (and update Trakt) my latest watched shows are kept updated on database.

***
### Refresh TMDB Data  
The python module [tmdb_manager.py](../scheduler/tmdb_manager.py) is responsible for getting the latest TMBB data from the API and save it to the database.  
It does a full refresh of all TMDB tables once a week (as it can take over 5 hours to complete).  
On a Daily basis, it compares current TMDB Data with latest Trakt Data and adds data for new shows.  

It is responsible for ...
* extracting, transforming, and loading detailed TV show data from the TMDb API into the database. (either full or delta), 
* calls the TMDb API to gather hierarchical data including shows, seasons, episodes, cast, crew, networks, artwork, and person details
* stores this data in structured lists while avoiding duplicates. 
* performs bulk inserts into the database via the DAO layer. 

The class also incorporates logging, error handling, and filtering logic (e.g. artwork language and key crew roles) to ensure clean, efficient data ingestion